<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [149]:
import pandas as pd
import matplotlib.pyplot as plt



In [150]:
df = pd.read_csv("smurph61_project_summary.csv", sep=";")
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

,project,commit,author,time,message,Month
0,easystats_parameters,0000296a2d6ee34cf9e717583df06a38e7842062,Daniel <mail@danielluedecke.de>,1684657123,Bug: `parameters` no longer respects the `vcov...,2023-05
1,easystats_parameters,000349ee3bbfe4b011a29a30f6d47a7a9486f5f6,GitHub Actions <actions@github.com>,1616069558,Built site for parameters: 0.12.0.1@14c1600\n,2021-03
2,easystats_parameters,00059db8ebf302fd696ae8a99df3b39ac57ea737,Daniel <mail@danielluedecke.de>,1603799266,save object name\n,2020-10
3,easystats_parameters,00084e7cfa0e749b92cca233453e6c5483ac5097,Daniel <mail@danielluedecke.de>,1600375007,fix check issues\n,2020-09
4,easystats_parameters,000b5bf7ed9e371fedc9625f78dd828d01f615d7,Daniel <mail@danielluedecke.de>,1659254294,lint\n,2022-07


In [151]:
projects = [
    'easystats_parameters',
    'ds4dm_tulip.jl',
    'mtiller_modelicabook',
    'jdtuck_fdasrvf_r',
    'xbarin02_collatz',
    'quantum-tii_qibo',
    'jklimke_libcitygml',
    'mskcc_vcf2maf',
    'sibrun_gh',
    'lyc102_ifem'
]

project_stats = []

for prj in projects:

    df1 = df[df['project'] == prj]

    df2 = df1.groupby('Month').size().reset_index(name='Commits')
    monthly_counts = df2
    monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'])
    full_date_range = pd.DataFrame({'time': pd.date_range(start=monthly_counts['time'].min(),end=monthly_counts['time'].max(),freq='MS')})
    monthly_counts = pd.merge(full_date_range, monthly_counts, on='time', how='left').fillna(0)
    monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')

    # Save monthly CSV
    monthly_counts[['Month', 'Commits']].rename(
        columns={'Commits': '#Commits'}
    ).to_csv(
        "smurph61_commits_timeseries_" + prj + ".csv",
        sep=";",
        index=False
    )
    zero_months = monthly_counts["Commits"] == 0
    
    gap_groups = (zero_months != zero_months.shift()).cumsum()
    
    gaps = monthly_counts[zero_months].groupby(gap_groups).agg(
        LongestGapStart=("Month", "first"),
        LongestGapEnd=("Month", "last"),
        LongestGapLength=("Month", "size")
    )
    
    if len(gaps) > 0:
        longest_gap = gaps.loc[gaps["LongestGapLength"].idxmax()]
    
        longest_gap_start = longest_gap["LongestGapStart"]
        longest_gap_end = longest_gap["LongestGapEnd"]
        longest_gap_length = int(longest_gap["LongestGapLength"])

        num_commits_after_gap = monthly_counts[monthly_counts["Month"] > longest_gap_end]["Commits"].sum()
    else:
        longest_gap_start = "N/A"
        longest_gap_end = "N/A"
        longest_gap_length = 0
        num_commits_after_gap = 0

    number_of_gaps = len(gaps[gaps["LongestGapLength"] >= 3])
    
    print(prj)
    print("Longest Gap Start:", longest_gap_start)
    print("Longest Gap End:", longest_gap_end)
    print("Longest Gap Length:", longest_gap_length)
    print("Commits After Gap:", int(num_commits_after_gap))
    print("Number of Gaps >= 3 Months:", number_of_gaps)
    print()

    # Create plot
    plt.figure(figsize=(10, 5))
    plt.plot(
        monthly_counts['Month'],
        monthly_counts['Commits'],
        marker='o',
        linestyle='-'
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(prj)
    plt.grid(True)
    plt.xticks(rotation=45, fontsize=6)
    plt.tight_layout()

    plt.savefig("smurph61_timeseries_" + prj + ".png",dpi=300 )

    plt.close()

    project_stats.append({
        "Project": prj,
        "LongestGapStart": longest_gap_start,
        "LongestGapEnd": longest_gap_end,
        "LongestGapLength": longest_gap_length,
        "NumCommitsAfterLastGap": int(num_commits_after_gap),
        "NumberOfGapsInTimeline": number_of_gaps,
        "ActivityPattern": ""
    })
    

easystats_parameters
Longest Gap Start: N/A
Longest Gap End: N/A
Longest Gap Length: 0
Commits After Gap: 0
Number of Gaps >= 3 Months: 0

ds4dm_tulip.jl
Longest Gap Start: 2024-07
Longest Gap End: 2024-11
Longest Gap Length: 5
Commits After Gap: 32
Number of Gaps >= 3 Months: 3

mtiller_modelicabook
Longest Gap Start: 2022-01
Longest Gap End: 2022-09
Longest Gap Length: 9
Commits After Gap: 144
Number of Gaps >= 3 Months: 12

jdtuck_fdasrvf_r
Longest Gap Start: 2014-08
Longest Gap End: 2015-10
Longest Gap Length: 15
Commits After Gap: 847
Number of Gaps >= 3 Months: 10

xbarin02_collatz
Longest Gap Start: 2024-04
Longest Gap End: 2024-11
Longest Gap Length: 8
Commits After Gap: 56
Number of Gaps >= 3 Months: 5

quantum-tii_qibo
Longest Gap Start: N/A
Longest Gap End: N/A
Longest Gap Length: 0
Commits After Gap: 0
Number of Gaps >= 3 Months: 0

jklimke_libcitygml
Longest Gap Start: 2020-12
Longest Gap End: 2021-09
Longest Gap Length: 10
Commits After Gap: 305
Number of Gaps >= 3 Months

In [152]:
activity_patterns = {
    'easystats_parameters': 'declining',
    'ds4dm_tulip.jl': 'declining',
    'mtiller_modelicabook': 'declining',
    'jdtuck_fdasrvf_r': 'irregular',
    'xbarin02_collatz': 'declining',
    'quantum-tii_qibo': 'irregular',
    'jklimke_libcitygml': 'U-shaped',
    'mskcc_vcf2maf': 'declining',
    'sibrun_gh': 'irregular',
    'lyc102_ifem': 'declining'
}

In [153]:
github_stats = {
    'easystats_parameters': [499, 46, '2026-10-05'],
    'ds4dm_tulip.jl': [163, 24, '2026-04-19'],
    'mtiller_modelicabook': [117, 75, '2026-09-06'],
    'jdtuck_fdasrvf_r': [15, 18, '2026-09-30'],
    'xbarin02_collatz': [84, 7, '2026-10-05'],
    'quantum-tii_qibo': [366, 104, '2026-10-07'],
    'jklimke_libcitygml': [111, 59, '2025-11-25'],
    'mskcc_vcf2maf': [421, 219, '2026-05-25'],
    'sibrun_gh': [2, 3, '2026-03-11'],
    'lyc102_ifem': [297, 99, '2026-08-19']
}


In [154]:
stats_df = pd.DataFrame(project_stats)

for i, row in stats_df.iterrows():
    prj = row["Project"]
    project_df = df[df["project"] == prj]

    stats_df.loc[i, "ncommits"] = project_df["commit"].nunique()
    stats_df.loc[i, "nauthors"] = project_df["author"].nunique()
    stats_df.loc[i, "from"] = project_df["time"].min()
    stats_df.loc[i, "to"] = project_df["time"].max()

    stats_df.loc[i, "nstars"] = github_stats[prj][0]
    stats_df.loc[i, "nforks"] = github_stats[prj][1]
    stats_df.loc[i, "lastGHCommitDate"] = github_stats[prj][2]

    stats_df.loc[i, "ActivityPattern"] = activity_patterns[prj]

stats_df = stats_df[
    [
        "Project",
        "ncommits",
        "nauthors",
        "from",
        "to",
        "nstars",
        "nforks",
        "lastGHCommitDate",
        "LongestGapStart",
        "LongestGapEnd",
        "LongestGapLength",
        "NumCommitsAfterLastGap",
        "ActivityPattern",
        "NumberOfGapsInTimeline"
    ]
]

stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)

stats_df

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline
0,easystats_parameters,8925.0,76.0,1.549685e+09,1.776170e+09,499.0,46.0,2026-10-05,N/A,N/A,0,0,declining,0
1,ds4dm_tulip.jl,926.0,29.0,1.524835e+09,1.759339e+09,163.0,24.0,2026-04-19,2024-07,2024-11,5,32,declining,3
2,mtiller_modelicabook,2558.0,69.0,1.354808e+09,1.742474e+09,117.0,75.0,2026-09-06,2022-01,2022-09,9,144,declining,12
3,jdtuck_fdasrvf_r,891.0,25.0,1.347235e+09,1.760872e+09,15.0,18.0,2026-09-30,2014-08,2015-10,15,847,irregular,10
4,xbarin02_collatz,1462.0,4.0,1.564771e+09,1.762346e+09,84.0,7.0,2026-10-05,2024-04,2024-11,8,56,declining,5
5,quantum-tii_qibo,14496.0,137.0,1.582014e+09,1.762324e+09,366.0,104.0,2026-10-07,N/A,N/A,0,0,irregular,0
6,jklimke_libcitygml,598.0,58.0,1.349084e+09,1.760909e+09,111.0,59.0,2025-11-25,2020-12,2021-09,10,305,U-shaped,9
7,mskcc_vcf2maf,498.0,42.0,1.385144e+09,1.754097e+09,421.0,219.0,2026-05-25,2021-05,2022-05,13,36,declining,5
8,sibrun_gh,1032.0,11.0,1.511975e+09,1.759783e+09,2.0,3.0,2026-03-11,2018-08,2021-03,32,700,irregular,5
9,lyc102_ifem,359.0,14.0,1.559805e+09,1.760729e+09,297.0,99.0,2026-08-19,2023-03,2023-10,8,22,declining,4


# Interpretation

### easystats_parameters
The activity pattern is declining. Commit activity is high in the earlier years but generally decreases over time. The project has 0 inactivity gaps of at least three months.

### ds4dm_tulip.jl
The activity pattern is declining. The project has much higher activity early in its history, followed by lower and less frequent activity in later years. Its longest inactivity gap occurred from 2024-07 to 2024-11 and lasted 5 months. The project has 3 gaps of at least three months.

### mtiller_modelicabook
The activity pattern is declining. The project has frequent bursts of activity in early year, while later years have significantly less spikes. Its longest inactivity gap occurred from 2022-01 to 2022-09 and lasted 9 months. The project has 12 gaps of at least three months.

### jdtuck_fdasrvf_r
The activity pattern is irregular. Development occurs in uneven bursts separated by periods of little or no activity. It has no consistent upward or downward trend. Its longest inactivity gap occurred from 2014-08 to 2015-10 and lasted 15 months. The project has 10 gaps of at least three months.

### xbarin02_collatz
The activity pattern is declining. The project begins with extremely high commit activity. Activity declines sharply and remains low for most of the later timeline. Its longest inactivity gap occurred from 2024-04 to 2024-11 and lasted 8 months. The project has 5 gaps of at least three months.

### quantum-tii_qibo
The activity pattern is irregular. The project remains active throughout the timeline, however commit counts are not consistant can contain many spikes. There are no inactivity gaps in the monthly timeline, so the number of gaps of at least three months is 0.

### jklimke_libcitygml
The activity pattern is U-shaped. Activity is high early in the timeline, declines to very low levels through the middle period, and then increases again toward the end. Its longest inactivity gap occurred from 2020-12 to 2021-09 and lasted 10 months. The project has 9 gaps of at least three months.

### mskcc_vcf2maf
The activity pattern is declining. Earlier years show consistent commit activity. Later years show  inactive periods and only occasional small bursts. Its longest inactivity gap occurred from 2021-05 to 2022-05 and lasted 13 months. The project has 5 gaps of at least three months.

### sibrun_gh
The activity pattern is irregular. The project contains several large bursts of commits with long periods with little or no activity in between. Its longest inactivity gap occurred from 2018-08 to 2021-03 and lasted 32 months. The project has 5 gaps of at least three months.

### lyc102_ifem
The activity pattern is declining. The highest levels of activity occur early in the project, followed by consistently low activity. Its longest inactivity gap occurred from 2023-03 to 2023-10 and lasted 8 months. The project has 4 gaps of at least three months.


In [155]:
gap_commits = []

for _, stats_row in stats_df.iterrows():
    prj = stats_row["Project"]

    gap_start = stats_row["LongestGapStart"]
    gap_end = stats_row["LongestGapEnd"]

    # Skip projects that had no inactivity gap
    if gap_start == "N/A" or gap_end == "N/A":
        print(prj, "- no inactivity gap")
        continue

    # Get only this project's commits
    project_df = df[df["project"] == prj].copy()

    # Convert commit time to month
    project_df["CommitMonth"] = pd.to_datetime(
        project_df["time"], unit="s"
    ).dt.to_period("M")

    gap_start_period = pd.Period(gap_start, freq="M")
    gap_end_period = pd.Period(gap_end, freq="M")

    # Last 10 
    pre_commits = (
        project_df[project_df["CommitMonth"] < gap_start_period]
        .sort_values("time", ascending=False)
        .head(10)
        .sort_values("time")
    )

    # First 10 
    post_commits = (
        project_df[project_df["CommitMonth"] > gap_end_period]
        .sort_values("time")
        .head(10)
    )

    for _, commit in pre_commits.iterrows():
        gap_commits.append({
            "pre/post": "pre",
            "prj": prj,
            "commit": commit["commit"],
            "author": commit["author"],
            "time": commit["time"],
            "message": commit["message"]
        })

    for _, commit in post_commits.iterrows():
        gap_commits.append({
            "pre/post": "post",
            "prj": prj,
            "commit": commit["commit"],
            "author": commit["author"],
            "time": commit["time"],
            "message": commit["message"]
        })

easystats_parameters - no inactivity gap
quantum-tii_qibo - no inactivity gap


In [156]:
gap_commits_df = pd.DataFrame(gap_commits)
gap_commits_df.groupby(
    ["prj", "pre/post"]
).size()

prj                   pre/post
ds4dm_tulip.jl        post        10
                      pre         10
jdtuck_fdasrvf_r      post        10
                      pre         10
jklimke_libcitygml    post        10
                      pre         10
lyc102_ifem           post        10
                      pre         10
mskcc_vcf2maf         post        10
                      pre         10
mtiller_modelicabook  post        10
                      pre         10
sibrun_gh             post        10
                      pre         10
xbarin02_collatz      post        10
                      pre         10
dtype: int64

In [157]:
gap_commits_df.to_csv(
    "smurph61_project_gap_commits.csv",
    index=False
)

In [158]:
for prj in gap_commits_df["prj"].unique():

    print("\n==============================")
    print(prj)
    print("==============================")

    print("\nPRE-GAP COMMITS:")
    pre = gap_commits_df[
        (gap_commits_df["prj"] == prj) &
        (gap_commits_df["pre/post"] == "pre")
    ]

    for msg in pre["message"]:
        print("-", msg)

    print("\nPOST-GAP COMMITS:")
    post = gap_commits_df[
        (gap_commits_df["prj"] == prj) &
        (gap_commits_df["pre/post"] == "post")
    ]

    for msg in post["message"]:
        print("-", msg)


ds4dm_tulip.jl

PRE-GAP COMMITS:
- build based on 2edf525

- Merge 9bbe799dcb7c99ea0fdede735d97e466e68d02a3 into 2edf525caf73b16c1f0b2da39e300a64aacb2b63

- Add a function that was promised in the documentation: https://ds4dm.github.io/Tulip.jl/dev/reference/attributes/
> SolutionTime	Float64	Solution time, in seconds

- Allow querying SolutionTime attribute (#146)

> SolutionTime	Float64	Solution time, in seconds
- build based on 8a3cce9

- Merge 9bbe799dcb7c99ea0fdede735d97e466e68d02a3 into 8a3cce91823bb23e81341aa78218504ef3cccf6f

- Count another location of `KKT.solve!()` call in the final stats
(previously unreported).

- Count another location of `KKT.solve!()` call in the final stats (#147)

(previously unreported).
- build based on f3df04d

- Merge 9bbe799dcb7c99ea0fdede735d97e466e68d02a3 into f3df04d8f88cdf2466b886d117e8d28ca537b4a1


POST-GAP COMMITS:
- Modify behavior of primal/dual objval attribute

- Enable unbounded MOI tests

- Update Tulip example tests

- Fix dual obj

In [159]:
themes = {
    "easystats_parameters": {
        "before": "N/A",
        "after": "N/A"
    },

    "ds4dm_tulip.jl": {
        "before": "Feature development:Bug fixes",
        "after": "Bug fixes:Dependency updates"
    },

    "mtiller_modelicabook": {
        "before": "Documentation updates:Other",
        "after": "Bug fixes:Dependency updates"
    },

    "jdtuck_fdasrvf_r": {
        "before": "Feature development:Bug fixes",
        "after": "Feature development:Bug fixes"
    },

    "xbarin02_collatz": {
        "before": "Feature development:Other",
        "after": "Feature development:Documentation updates"
    },

    "quantum-tii_qibo": {
        "before": "N/A",
        "after": "N/A"
    },

    "jklimke_libcitygml": {
        "before": "Feature development:Bug fixes",
        "after": "Feature development:Bug fixes"
    },

    "mskcc_vcf2maf": {
        "before": "Documentation updates:Bug fixes",
        "after": "Feature development:Bug fixes"
    },

    "sibrun_gh": {
        "before": "Documentation updates:Other",
        "after": "Feature development:Bug fixes"
    },

    "lyc102_ifem": {
        "before": "Feature development:Bug fixes",
        "after": "Bug fixes:Feature development"
    }
}

In [160]:
stats_df["BeforeThemes"] = ""
stats_df["AfterThemes"] = ""

for i, row in stats_df.iterrows():
    prj = row["Project"]

    stats_df.loc[i, "BeforeThemes"] = themes[prj]["before"]
    stats_df.loc[i, "AfterThemes"] = themes[prj]["after"]

stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)
# Check
stats_df[
    ["Project", "BeforeThemes", "AfterThemes"]
]

,Project,BeforeThemes,AfterThemes
0,easystats_parameters,N/A,N/A
1,ds4dm_tulip.jl,Feature development:Bug fixes,Bug fixes:Dependency updates
2,mtiller_modelicabook,Documentation updates:Other,Bug fixes:Dependency updates
3,jdtuck_fdasrvf_r,Feature development:Bug fixes,Feature development:Bug fixes
4,xbarin02_collatz,Feature development:Other,Feature development:Documentation updates
5,quantum-tii_qibo,N/A,N/A
6,jklimke_libcitygml,Feature development:Bug fixes,Feature development:Bug fixes
7,mskcc_vcf2maf,Documentation updates:Bug fixes,Feature development:Bug fixes
8,sibrun_gh,Documentation updates:Other,Feature development:Bug fixes
9,lyc102_ifem,Feature development:Bug fixes,Bug fixes:Feature development


In [161]:
gap_interpretations = {
    "easystats_parameters": {
        "HypothesizedGapReason": "N/A active Project",
        "HasRecovered": "N/A",
        "WhyRecovered": "N/A",
    },

    "ds4dm_tulip.jl": {
        "HypothesizedGapReason": "Development appears to have paused after a period of solver feature and statistics work.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed with fixes to objective-value handling, tests, CI, and package dependencies.",
    },

    "mtiller_modelicabook": {
        "HypothesizedGapReason": "Development appears to have slowed after documentation, build maintenance, and submodule updates.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed to fix model and build compatibility issues and modernize support features.",
    },

    "jdtuck_fdasrvf_r": {
        "HypothesizedGapReason": "The gap followed completion and debugging of a major regression feature set, natural after a large milestone.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Development restarted with Rcpp conversion, CI work, new algorithms and functions, and bug fixes.",
    },

    "xbarin02_collatz": {
        "HypothesizedGapReason": "Activity appears to have paused after substantial computing-script development.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed with new HPC configuration, search functionality, documentation, and worker features.",
    },

    "quantum-tii_qibo": {
        "HypothesizedGapReason": "N/A active Project",
        "HasRecovered": "N/A",
        "WhyRecovered": "N/A",
        
    },

    "jklimke_libcitygml": {
        "HypothesizedGapReason": "Development paused after build, platform, and feature work.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Contributions restarted around implementation of DEM, ReliefComponent, RectifiedGridCoverage, and ExternalReference support.",
    },

    "mskcc_vcf2maf": {
        "HypothesizedGapReason": "The project appears to have shifted toward slower maintenance after documentation and API cleanup.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed mainly to maintain compatibility with newer VEP versions and address specific failures.",
    },

    "sibrun_gh": {
        "HypothesizedGapReason": "The project appears to have become dormant after mostly documentation and naming cleanup.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Development restarted with Python 3 adaptations, new graph functionality, display routines, and bug fixes.",
    },

    "lyc102_ifem": {
        "HypothesizedGapReason": "Development temporarily slowed after project documentation, testing work, MATLAB compatibility fixes.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Work resumed to fix bugs and add new numerical and project functionality.",
    }
}

In [162]:
stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)

In [163]:
for prj in gap_commits_df["prj"].unique():

    pre_authors = set(
        gap_commits_df[
            (gap_commits_df["prj"] == prj) &
            (gap_commits_df["pre/post"] == "pre")
        ]["author"]
    )

    post_authors = set(
        gap_commits_df[
            (gap_commits_df["prj"] == prj) &
            (gap_commits_df["pre/post"] == "post")
        ]["author"]
    )

    same_authors = pre_authors & post_authors
    new_authors = post_authors - pre_authors

    print("\n", prj)
    print("Same authors after gap:", same_authors)
    print("New authors after gap:", new_authors)


 ds4dm_tulip.jl
Same authors after gap: {'Documenter.jl <documenter@juliadocs.github.io>'}
New authors after gap: {'mtanneau <mathieu.tanneau@gmail.com>', 'Mathieu Tanneau <9593025+mtanneau@users.noreply.github.com>'}

 mtiller_modelicabook
Same authors after gap: {'dependabot[bot] <49699333+dependabot[bot]@users.noreply.github.com>', 'Michael Tiller <michael.tiller@gmail.com>'}
New authors after gap: {'ldwoolley <ldwoolley@gmail.com>', 'root <eraracom@gmail.com>'}

 jdtuck_fdasrvf_r
Same authors after gap: {'J. Derek Tucker <jdtuck@sandia.gov>'}
New authors after gap: {'Derek Tucker <jdtuck@sandia.gov>'}

 xbarin02_collatz
Same authors after gap: {'David Barina <ibarina@fit.vutbr.cz>'}
New authors after gap: set()

 jklimke_libcitygml
Same authors after gap: {'Jan Klimke <jklimke@users.noreply.github.com>'}
New authors after gap: {'HarutakaMatsumoto <harutaka_matsumoto@icloud.com>'}

 mskcc_vcf2maf
Same authors after gap: {'Alan Hoyle <alan@alanhoyle.com>'}
New authors after gap: {'b

In [164]:
who_recovered = {
    "easystats_parameters": "N/A",
    
    "ds4dm_tulip.jl": "Same and new contributors",
    
    "mtiller_modelicabook": "Same and new contributors",
    
    "jdtuck_fdasrvf_r": "Same contributor",
    
    "xbarin02_collatz": "Same contributor",
    
    "quantum-tii_qibo": "N/A",
    
    "jklimke_libcitygml": "Same and new contributors",
    
    "mskcc_vcf2maf": "Same and new contributors",
    
    "sibrun_gh": "Same and new contributors",
    
    "lyc102_ifem": "Same and new contributors"
}

In [165]:
stats_df["WhoRecovered"] = ""

for i, row in stats_df.iterrows():
    prj = row["Project"]
    stats_df.loc[i, "WhoRecovered"] = who_recovered[prj]

In [166]:
stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)


In [167]:
# Check
stats_df[
    ["Project", "WhoRecovered"]
]

,Project,WhoRecovered
0,easystats_parameters,N/A
1,ds4dm_tulip.jl,Same and new contributors
2,mtiller_modelicabook,Same and new contributors
3,jdtuck_fdasrvf_r,Same contributor
4,xbarin02_collatz,Same contributor
5,quantum-tii_qibo,N/A
6,jklimke_libcitygml,Same and new contributors
7,mskcc_vcf2maf,Same and new contributors
8,sibrun_gh,Same and new contributors
9,lyc102_ifem,Same and new contributors


In [168]:
stats_df["HypothesizedGapReason"] = ""
stats_df["HasRecovered"] = ""
stats_df["WhyRecovered"] = ""
stats_df["WhoRecovered"] = ""

for i, row in stats_df.iterrows():
    prj = row["Project"]

    stats_df.loc[i, "HypothesizedGapReason"] = gap_interpretations[prj]["HypothesizedGapReason"]
    stats_df.loc[i, "HasRecovered"] = gap_interpretations[prj]["HasRecovered"]
    stats_df.loc[i, "WhyRecovered"] = gap_interpretations[prj]["WhyRecovered"]
    stats_df.loc[i, "WhoRecovered"] = who_recovered[prj]
    

In [169]:
stats_df[
    [
        "Project",
        "HypothesizedGapReason",
        "HasRecovered",
        "WhyRecovered",
        "WhoRecovered",
    ]
]

,Project,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered
0,easystats_parameters,N/A active Project,N/A,N/A,N/A
1,ds4dm_tulip.jl,Development appears to have paused after a per...,Yes,Work resumed with fixes to objective-value han...,Same and new contributors
2,mtiller_modelicabook,Development appears to have slowed after docum...,Yes,Work resumed to fix model and build compatibil...,Same and new contributors
3,jdtuck_fdasrvf_r,The gap followed completion and debugging of a...,Yes,"Development restarted with Rcpp conversion, CI...",Same contributor
4,xbarin02_collatz,Activity appears to have paused after substant...,Yes,"Work resumed with new HPC configuration, searc...",Same contributor
5,quantum-tii_qibo,N/A active Project,N/A,N/A,N/A
6,jklimke_libcitygml,"Development paused after build, platform, and ...",Yes,Contributions restarted around implementation ...,Same and new contributors
7,mskcc_vcf2maf,The project appears to have shifted toward slo...,Yes,Work resumed mainly to maintain compatibility ...,Same and new contributors
8,sibrun_gh,The project appears to have become dormant aft...,Yes,Development restarted with Python 3 adaptation...,Same and new contributors
9,lyc102_ifem,Development temporarily slowed after project d...,Yes,Work resumed to fix bugs and add new numerical...,Same and new contributors


In [170]:
stats_df["Currentstatus"] = "Active"

In [171]:
recent_commits = []

for prj in df["project"].unique():

    project_df = df[df["project"] == prj].copy()

    latest = (
        project_df
        .sort_values("time", ascending=False)
        .head(10)
    )

    for _, commit in latest.iterrows():
        recent_commits.append({
            "prj": prj,
            "commit": commit["commit"],
            "author": commit["author"],
            "time": commit["time"],
            "message": commit["message"]
        })

recent_commits_df = pd.DataFrame(recent_commits)

In [172]:
for prj in recent_commits_df["prj"].unique():

    print("=============")
    print(prj)
    print("=============")
 

    project_recent = recent_commits_df[
        recent_commits_df["prj"] == prj
    ]

    for msg in project_recent["message"]:
        print("-", msg)

easystats_parameters
- Deploying to gh-pages from @ easystats/parameters@ca2001e2cbfca09a6613da0ec9ffb4fb898e2904 🚀

- Merge 391c34643c5b42b73f3a42957a6e88fe699a4b23 into ca2001e2cbfca09a6613da0ec9ffb4fb898e2904

- Merge 7436d67641ef1ef8e7a242d5b49a5063693601f1 into ca2001e2cbfca09a6613da0ec9ffb4fb898e2904

- Merge ec0c32b296f5dfe6f82b0c2a01f1ae07fc7667eb into ca2001e2cbfca09a6613da0ec9ffb4fb898e2904

- Merge 0cfa2801e5e3bf7935c5a26a53c11a1a8efdc81f into ca2001e2cbfca09a6613da0ec9ffb4fb898e2904

- fix

- Show `plot()` usage in `n_factors()` examples (#1205)

* Initial plan

* docs: show plot example for n_factors

Agent-Logs-Url: https://github.com/easystats/parameters/sessions/ffe1fac9-e7bb-45d7-be07-e065b5a504f7

Co-authored-by: DominiqueMakowski <8875533+DominiqueMakowski@users.noreply.github.com>

* fix

---------

Co-authored-by: copilot-swe-agent[bot] <198982749+Copilot@users.noreply.github.com>
Co-authored-by: DominiqueMakowski <8875533+DominiqueMakowski@users.noreply.github.com

In [173]:
recent_themes = {
    "easystats_parameters": "Documentation updates:Automated bot contributions",

    "ds4dm_tulip.jl": "Feature development:Release",

    "mtiller_modelicabook": "Other:Dependency updates",

    "jdtuck_fdasrvf_r": "Bug fixes:Release",

    "xbarin02_collatz": "Feature development:Bug fixes",

    "quantum-tii_qibo": "Feature development:Bug fixes",

    "jklimke_libcitygml": "Bug fixes:Feature development",

    "mskcc_vcf2maf": "Bug fixes:Feature development",

    "sibrun_gh": "Other:Other",

    "lyc102_ifem": "Bug fixes:Feature development"
}

In [174]:
stats_df["RecentThemes"] = ""

for i, row in stats_df.iterrows():
    prj = row["Project"]
    stats_df.loc[i, "RecentThemes"] = recent_themes[prj]

In [175]:
stats_df.to_csv(
    "smurph61_project_stats.csv",
    index=False
)

In [176]:
stats_df[
    ["Project", "Currentstatus", "RecentThemes"]
]

,Project,Currentstatus,RecentThemes
0,easystats_parameters,Active,Documentation updates:Automated bot contributions
1,ds4dm_tulip.jl,Active,Feature development:Release
2,mtiller_modelicabook,Active,Other:Dependency updates
3,jdtuck_fdasrvf_r,Active,Bug fixes:Release
4,xbarin02_collatz,Active,Feature development:Bug fixes
5,quantum-tii_qibo,Active,Feature development:Bug fixes
6,jklimke_libcitygml,Active,Bug fixes:Feature development
7,mskcc_vcf2maf,Active,Bug fixes:Feature development
8,sibrun_gh,Active,Other:Other
9,lyc102_ifem,Active,Bug fixes:Feature development


In [177]:
reflections = {
    "easystats_parameters":
        "This project did not have a meaningful inactivity gap, so I cannot identify a specific point to explain. This project’s activity has been steadily declining, with fewer commits to the repository. Its most recent commits were related to documentation or bot activity, demonstrating that it still receives some upkeep.",

    "ds4dm_tulip.jl":
        "The longest inactivity gap in this project’s history was five months. It was moderately easy to explain because the commits before and after the gap differed significantly. Before the gap, the project involved mainly solver features and statistics. In contrast, the commits after the gap contained mainly bug fixes, tests, CI, and dependencies. This project recovered well from the gap because both previous contributors and new ones participated in the commits, which involved features and release.",

    "mtiller_modelicabook":
        "The longest inactivity gap in this project’s history was nine months. The reason for this gap was not entirely clear because the commits before and after the gap involved documentation, build maintenance, merging, and submodules. However, the project team worked on compatibility, dependencies, and modernization after the gap, possibly relating to the switch from Python 2 to Python 3. The project maintained an excellent level of upkeep even after the long inactivity gap. The same contributor worked on different aspects of the project before and after the gap. The later commits involved Rcpp, algorithms, CI, and bug fixes.",
    "jdtuck_fdasrvf_r":
        "The longest inactivity gap in this project’s history was fifteen months. It occurred right after commits involving a significant block of regression functionality, which might signify the completion of a particular stage of development. This information suggests that the inactivity gap can be seen as a pause between certain parts of the project’s development. The project appeared to be well maintained before and after the gap. The same contributor worked on the file RcppExports, and new commits involved modernization, Python, and OpenModelica.",
    "xbarin02_collatz":
        "The longest inactivity gap in this project’s history was eight months. The reason for this gap was unclear because the descriptions of the commits before and after the gap did not involve significant changes and mostly included documentation, building, and submodules. It can be speculated that the long inactivity gap was caused by the completion of a part of the development process. The same contributor worked on the project before and after the gap. Later commits involved new search functions, configuration, documentation, and workers.",

    "quantum-tii_qibo":
        "This project did not have a meaningful inactivity gap, so I cannot identify a point to explain. This project’s activity has been steadily declining, with fewer commits to the repository, and its most recent commits were related to new functionality, tests and dependencies, and bugs.",
    "jklimke_libcitygml":
        "The longest inactivity gap in this project’s history was ten months. The reason for this gap was unclear because the descriptions of the commits before and after the gap did not involve significant changes and mostly included documentation and command-line clean-up. However, the project team appeared to recover from the inactivity gap well because the later commits involved significant changes in multiple files and utilized multiple contributors.",
    "mskcc_vcf2maf":
        "The longest inactivity gap in this project’s history was thirteen months. This project appeared to be slowly abandoned because the commits before the gap included primarily documentation and command-line clean-up. The project team appeared to recover from the inactivity gap moderately well, addressing various compatibility issues, VEP options, and bugs, and utilizing both previous contributors and new ones.",

    "sibrun_gh":
        "The longest inactivity gap in this project’s history was thirty two months and was the largest of all gaps. It was challenging to determine the reason for this gap because the commits before and after the gap included primarily documentation and name changes. However, the later commits appear to involve more significant changes, such as Python 3 compatibility, graph, display, and bugs.",

    "lyc102_ifem":
        "The longest inactivity gap in this project’s history was eight months. The reason for this gap was unclear because the commits before and after the gap involved various changes, including documentation, tests, MATLAB, and features. However, the project team addressed multiple commits after the gap, including bugs and new numerical functions, features, and contributors."}

In [178]:
for prj, reflection in reflections.items():

    filename = "smurph61_reflection_" + prj + ".md"

    with open(filename, "w") as f:
        f.write("# Reflection: " + prj + "\n\n")
        f.write(reflection)

    stats_df.loc[
        stats_df["Project"] == prj,
        "Notes"
    ] = reflection

In [179]:
print(stats_df.columns.tolist())


['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'ActivityPattern', 'NumberOfGapsInTimeline', 'BeforeThemes', 'AfterThemes', 'WhoRecovered', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'Currentstatus', 'RecentThemes', 'Notes']
